# Empirical Example for Sensitivity Analysis: Gasoline Demand

This is an adaptation of the replication package for Chernozhukov, Cinelli, Newey, Sharma and Syrgkanis,
*Long Story Short: Omitted Variable Bias in Causal Machine Learning* 

The original replication package can be found __[here](https://dataverse.harvard.edu/dataset.xhtml?persistentId=doi:10.7910/DVN/VH6QHQ)__.

In [ ]:
!pip install econml

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy
import scipy.special
import scipy.stats
import joblib
from statsmodels.api import OLS
from sklearn.model_selection import StratifiedGroupKFold, KFold
import pandas as pd
import math
from sklearn.base import clone, BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LassoCV, LogisticRegressionCV, LinearRegression, Lasso, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, RandomForestClassifier, GradientBoostingClassifier
from scipy.optimize import fmin_l_bfgs_b
from econml.dml import LinearDML
from econml.sklearn_extensions.model_selection import GridSearchCVList
from econml.utilities import cross_product
# BaseGRF is an internal econml API; may need updating across econml versions
from econml.grf._base_grf import BaseGRF

In [ ]:
import warnings
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings('ignore', category=ConvergenceWarning)

## Module Definitions

The following cells define helper classes and functions used by this notebook.
They are inlined here so the notebook runs in Google Colab without requiring local files.

In [ ]:
def mean_ci(data, confidence=0.95):
    a = 1.0 * np.array(data).flatten()
    n = len(a)
    m, se = np.mean(a), scipy.stats.sem(a)
    h = se * scipy.stats.t.ppf((1 + confidence) / 2., n-1)
    return m, se, m-h, m+h

In [ ]:
class Featurizer(TransformerMixin, BaseEstimator):
    ''' Creates features of the form X[:, 1:] * poly(X[:, 0])
    i.e. some polynomial of the treatment, multiplied by some other feature.
    '''

    def __init__(self, *, degree=1, normalize=True):
        self.degree = degree
        self.normalize = normalize

    def _transform(self, X):
        Tfeats = PolynomialFeatures(degree=self.degree, include_bias=True).fit_transform(X[:, [0]])
        feats = cross_product(Tfeats, X[:, 1:])
        return feats

    def fit(self, X, y=None):
        if self.normalize:
            self.scaler_ = StandardScaler().fit(self._transform(X))
        return self

    def transform(self, X):
        if self.normalize:
            return self.scaler_.transform(self._transform(X))
        return self._transform(X)


class SieveRR(BaseEstimator):
    ''' Linear sieve representation learning with some feature map and l2 penalty. '''

    def __init__(self, *, featurizer, moment_fn, alpha=0):
        self.alpha = alpha
        self.featurizer = featurizer
        self.moment_fn = moment_fn

    def fit(self, X):
        self.featurizer_ = clone(self.featurizer).fit(X)
        n_samples = X.shape[0]
        Xfeats = self.featurizer_.transform(X)
        Xfeats = np.hstack([np.ones((Xfeats.shape[0], 1)), Xfeats])
        n_feats = Xfeats.shape[1]
        Mfeats = np.zeros((n_samples, n_feats))
        for t in np.arange(n_feats):
            if t == 0:
                feat_fn = lambda x: np.ones(x.shape[0])
            else:
                feat_fn = lambda x: self.featurizer_.transform(x)[:, t - 1]
            Mfeats[:, t] = self.moment_fn(X, feat_fn)
        Sigma = Xfeats.T @ Xfeats / n_samples
        reg = self.alpha * np.eye(n_feats)
        reg[0, 0] = 0
        Sigma += reg
        params = np.linalg.pinv(Sigma) @ np.mean(Mfeats, axis=0)
        self.coef_ = params[1:]
        self.intercept_ = params[0]
        return self

    def predict(self, X):
        return self.featurizer_.transform(X) @ self.coef_ + self.intercept_


class SparseSieveRR(BaseEstimator):
    ''' Linear sieve representation learning with l1+l2 (elastic net) penalty. '''

    def __init__(self, *, featurizer, moment_fn, alpha=0.0, l1_ratio=1.0, tol=1e-4):
        self.featurizer = featurizer
        self.moment_fn = moment_fn
        self.alpha = alpha
        self.l1_ratio = l1_ratio
        self.tol = tol

    def fit(self, X):
        self.featurizer_ = clone(self.featurizer).fit(X)
        n_samples = X.shape[0]
        Xfeats = self.featurizer_.transform(X)
        Xfeats = np.hstack([np.ones((Xfeats.shape[0], 1)), Xfeats])
        n_feats = Xfeats.shape[1]
        Mfeats = np.zeros((n_samples, n_feats))
        for t in np.arange(n_feats):
            if t == 0:
                feat_fn = lambda x: np.ones(x.shape[0])
            else:
                feat_fn = lambda x: self.featurizer_.transform(x)[:, t - 1]
            Mfeats[:, t] = self.moment_fn(X, feat_fn)
        Sigma = Xfeats.T @ Xfeats / n_samples
        mfeats = np.mean(Mfeats, axis=0)

        alpha_l1 = self.alpha * self.l1_ratio
        alpha_l2 = self.alpha * (1 - self.l1_ratio)

        def loss_and_jac(extended_coef):
            coef = extended_coef[:n_feats] - extended_coef[n_feats:]
            loss = - 2 * mfeats @ coef + coef.T @ Sigma @ coef
            loss += alpha_l1 * np.sum(extended_coef[1:n_feats])
            loss += alpha_l1 * np.sum(extended_coef[n_feats + 1:])
            loss += 0.5 * alpha_l2 * np.sum(extended_coef[1:n_feats]**2)
            loss += 0.5 * alpha_l2 * np.sum(extended_coef[n_feats + 1:]**2)
            grad = 2 * (Sigma @ coef - mfeats)
            jac = np.concatenate([grad, -grad])
            jac[1:n_feats] += alpha_l1 + alpha_l2 * extended_coef[1:n_feats]
            jac[n_feats + 1:] += alpha_l1 + alpha_l2 * extended_coef[n_feats + 1:]
            return loss, jac

        w, _, _ = fmin_l_bfgs_b(loss_and_jac, np.zeros(2 * n_feats),
                                 bounds=[(0, None)] * 2 * n_feats,
                                 pgtol=self.tol)
        params = w[:n_feats] - w[n_feats:]
        self.coef_ = params[1:]
        self.intercept_ = params[0]
        return self

    def predict(self, X):
        return self.featurizer_.transform(X) @ self.coef_ + self.intercept_

In [ ]:
def poly_feature_fns(degree):
    def poly(d, sign=1.0):
        return lambda x: sign * x[:, [0]]**d
    return [poly(t) for t in np.arange(0, degree + 1)]


class RFrr(BaseGRF):
    """
    A forest that estimates a Riesz representer of a linear functional.
    Minimizes the Riesz loss E[(phi(T,X)' alpha(x))^2 - 2 * <alpha(x), m(Z; phi)> | X=x].

    Parameters
    ----------
    riesz_feature_fns : list of lambda
        Features phi(T,X) used to approximate the Riesz representer locally.
    moment_fn : lambda (x, fn) -> float
        The linear moment function m([T;X], g).
    l2 : float, default=0.01
        L2 penalty on the local covariance matrix (no penalty on intercept).
    """

    def __init__(self, *, riesz_feature_fns, moment_fn, l2=0.01,
                 n_estimators=100, criterion='mse', max_depth=None,
                 min_samples_split=10, min_samples_leaf=5,
                 min_weight_fraction_leaf=0., min_var_fraction_leaf=None,
                 min_var_leaf_on_val=False, max_features='auto',
                 min_impurity_decrease=0., max_samples=.45,
                 min_balancedness_tol=.45, honest=True, inference=True,
                 fit_intercept=True, subforest_size=4, n_jobs=-1,
                 random_state=None, verbose=0, warm_start=False):
        self.riesz_feature_fns = riesz_feature_fns
        self.moment_fn = moment_fn
        self.l2 = l2
        super().__init__(n_estimators=n_estimators, criterion=criterion,
                         max_depth=max_depth, min_samples_split=min_samples_split,
                         min_samples_leaf=min_samples_leaf,
                         min_weight_fraction_leaf=min_weight_fraction_leaf,
                         min_var_fraction_leaf=min_var_fraction_leaf,
                         min_var_leaf_on_val=min_var_leaf_on_val,
                         max_features=max_features,
                         min_impurity_decrease=min_impurity_decrease,
                         max_samples=max_samples,
                         min_balancedness_tol=min_balancedness_tol,
                         honest=honest, inference=inference,
                         fit_intercept=False, subforest_size=subforest_size,
                         n_jobs=n_jobs, random_state=random_state,
                         verbose=verbose, warm_start=warm_start)

    def _get_alpha_and_pointJ(self, X, T, y):
        n_riesz_feats = len(self.riesz_feature_fns)
        TX = np.hstack([T, X])
        riesz_feats = np.hstack([feat_fn(TX) for feat_fn in self.riesz_feature_fns])
        mfeats = np.hstack([self.moment_fn(TX, feat_fn) for feat_fn in self.riesz_feature_fns])
        alpha = np.zeros((X.shape[0], n_riesz_feats))
        alpha[:, :n_riesz_feats] = mfeats
        riesz_cov_matrix = cross_product(riesz_feats, riesz_feats).reshape(
            (X.shape[0], n_riesz_feats, n_riesz_feats))
        penalty = self.l2 * np.eye(n_riesz_feats)
        penalty[0, 0] = 0
        pointJ = riesz_cov_matrix + penalty
        return alpha, pointJ.reshape((X.shape[0], -1))

    def _get_n_outputs_decomposition(self, X, T, y):
        n_relevant_outputs = len(self.riesz_feature_fns)
        return n_relevant_outputs, n_relevant_outputs

    def _translate(self, point, TX):
        riesz_feats = np.hstack([feat_fn(TX) for feat_fn in self.riesz_feature_fns])
        return np.sum(point[:, :riesz_feats.shape[1]] * riesz_feats, axis=1)

    def fit(self, X):
        return super().fit(X[:, 1:], X[:, [0]], X[:, [0]])

    def predict(self, X_test):
        point = super().predict(X_test[:, 1:])
        return self._translate(point, X_test)


class RFreg(BaseGRF):
    """
    A forest that estimates a regression function with pre-specified features.
    Minimizes E[(<theta(x), phi(T,X)> - Y)^2 | X=x].

    Parameters
    ----------
    reg_feature_fns : list of lambda
        Features phi(T,X) used to approximate the regression locally.
    l2 : float, default=0.01
        L2 penalty on the local covariance matrix (no penalty on intercept).
    """

    def __init__(self, *, reg_feature_fns, l2=0.01,
                 n_estimators=100, criterion='mse', max_depth=None,
                 min_samples_split=10, min_samples_leaf=5,
                 min_weight_fraction_leaf=0., min_var_fraction_leaf=None,
                 min_var_leaf_on_val=False, max_features='auto',
                 min_impurity_decrease=0., max_samples=.45,
                 min_balancedness_tol=.45, honest=True, inference=True,
                 fit_intercept=True, subforest_size=4, n_jobs=-1,
                 random_state=None, verbose=0, warm_start=False):
        self.reg_feature_fns = reg_feature_fns
        self.l2 = l2
        super().__init__(n_estimators=n_estimators, criterion=criterion,
                         max_depth=max_depth, min_samples_split=min_samples_split,
                         min_samples_leaf=min_samples_leaf,
                         min_weight_fraction_leaf=min_weight_fraction_leaf,
                         min_var_fraction_leaf=min_var_fraction_leaf,
                         min_var_leaf_on_val=min_var_leaf_on_val,
                         max_features=max_features,
                         min_impurity_decrease=min_impurity_decrease,
                         max_samples=max_samples,
                         min_balancedness_tol=min_balancedness_tol,
                         honest=honest, inference=inference,
                         fit_intercept=False, subforest_size=subforest_size,
                         n_jobs=n_jobs, random_state=random_state,
                         verbose=verbose, warm_start=warm_start)

    def fit(self, X, y):
        return super().fit(X[:, 1:], X[:, [0]], y)

    def _get_alpha_and_pointJ(self, X, T, y):
        n_reg_feats = len(self.reg_feature_fns)
        TX = np.hstack([T, X])
        reg_feats = np.hstack([feat_fn(TX) for feat_fn in self.reg_feature_fns])
        alpha = y.reshape(-1, 1) * reg_feats
        reg_cov_matrix = cross_product(reg_feats, reg_feats).reshape(
            (X.shape[0], n_reg_feats, n_reg_feats))
        penalty = self.l2 * np.eye(n_reg_feats)
        penalty[0, 0] = 0
        pointJ = reg_cov_matrix + penalty
        return alpha, pointJ.reshape((X.shape[0], -1))

    def _get_n_outputs_decomposition(self, X, T, y):
        n_relevant_outputs = len(self.reg_feature_fns)
        return n_relevant_outputs, n_relevant_outputs

    def _translate(self, point, TX):
        reg_feats = np.hstack([feat_fn(TX) for feat_fn in self.reg_feature_fns])
        return np.sum(point * reg_feats, axis=1)

    def predict(self, X_test):
        point = super().predict(X_test[:, 1:])
        return self._translate(point, X_test)

In [ ]:
class DebiasedMoment:

    def __init__(self, *, moment_fn, get_reisz_fn, get_reg_fn, n_splits=5):
        self.moment_fn = moment_fn
        self.get_reisz_fn = get_reisz_fn
        self.get_reg_fn = get_reg_fn
        self.n_splits = n_splits

    def fit(self, X, y):
        y = y.flatten()
        if self.n_splits == 1:
            splits = [(np.arange(X.shape[0]), np.arange(X.shape[0]))]
        elif isinstance(self.n_splits, int):
            splits = list(KFold(n_splits=self.n_splits).split(X))
        else:
            splits = self.n_splits

        reisz_pred = np.zeros(X.shape[0])
        reg_pred = np.zeros(X.shape[0])
        moment_pred = np.zeros(X.shape[0])
        moment_reisz = np.zeros(X.shape[0])
        self.reisz_models_ = []
        self.reg_models_ = []
        reisz_fn = self.get_reisz_fn(X)
        reg_fn = self.get_reg_fn(X, y)
        for train, test in splits:
            reisz = reisz_fn().fit(X[train])
            reisz_pred[test] = reisz.predict(X[test])
            reg = reg_fn().fit(X[train], y[train])
            reg_pred[test] = reg.predict(X[test])
            moment_pred[test] = self.moment_fn(X[test], reg.predict)
            moment_reisz[test] = self.moment_fn(X[test], reisz.predict)
            self.reisz_models_.append(reisz)
            self.reg_models_.append(reg)

        self.reg_pred_ = reg_pred
        self.reisz_pred_ = reisz_pred
        self.moment_ = moment_pred + reisz_pred * (y - reg_pred)
        self.moment_reisz_ = moment_reisz
        self.moment_pred_ = moment_pred
        self.y_ = y.copy()

        epsilon = LinearRegression(fit_intercept=False).fit(
            reisz_pred.reshape(-1, 1), y - reg_pred).coef_[0]
        moment_pred = moment_pred + moment_reisz * epsilon
        reg_pred = reg_pred + epsilon * reisz_pred
        self.tmle_moment_ = moment_pred + reisz_pred * (y - reg_pred)
        return self

    def avg_moment(self, alpha=0.05, tmle=False):
        if tmle:
            return mean_ci(self.tmle_moment_, confidence=1 - alpha)
        return mean_ci(self.moment_, confidence=1 - alpha)

In [ ]:
def first_stage_reg(numeric_columns=[], cv=None):
    return GridSearchCVList([
        Pipeline([('scale', ColumnTransformer([('num', StandardScaler(), numeric_columns)],
                                               remainder='passthrough')),
                   ('lasso', Lasso())]),
        Pipeline([('scale', ColumnTransformer([('num', StandardScaler(), numeric_columns)],
                                               remainder='passthrough')),
                   ('feat', Featurizer(degree=1, normalize=False)),
                   ('lasso', Lasso())]),
        RandomForestRegressor(n_estimators=100, random_state=123),
        RFreg(reg_feature_fns=poly_feature_fns(10),
              min_var_leaf_on_val=True, min_impurity_decrease=1e-4,
              max_samples=.85, inference=False, subforest_size=1, random_state=123)
    ],
    param_grid_list=[
        {'lasso__alpha': [1e-6, 1e-5, 0.0001, .001, .01, .1, 1, 10]},
        {'lasso__alpha': [1e-6, 1e-5, 0.0001, .001, .01, .1, 1, 10]},
        {'max_depth': [3, 5], 'min_samples_leaf': [10, 50]},
        {'reg_feature_fns': [poly_feature_fns(2)], 'min_samples_leaf': [10],
         'min_var_fraction_leaf': [0.01, 0.1], 'l2': [1e-3, 1e-2]}
    ],
    cv=cv, scoring='neg_mean_squared_error', n_jobs=-1)


def reg_fn_gen(numeric_columns, cv=None, verbose=0):
    def get_reg_fn(X, y):
        est = first_stage_reg(numeric_columns, cv=cv).fit(X, y)
        if verbose > 0:
            print('Chose: ', est.best_score_, est.best_estimator_)
        return lambda: clone(est.best_estimator_)
    return get_reg_fn


def neg_reisz_score_gen(moment_fn):
    def neg_reisz_score(model, X):
        return np.mean(2 * moment_fn(X, model.predict) - model.predict(X)**2)
    return neg_reisz_score


def first_stage_rr(moment_fn, numeric_columns, cv=None):
    return GridSearchCVList([
        SieveRR(featurizer=Featurizer(degree=1), moment_fn=moment_fn),
        SieveRR(featurizer=Featurizer(degree=2), moment_fn=moment_fn),
        SparseSieveRR(featurizer=Featurizer(degree=2), moment_fn=moment_fn, tol=1e-4),
        SparseSieveRR(featurizer=Featurizer(degree=1), moment_fn=moment_fn, tol=1e-4),
        RFrr(riesz_feature_fns=poly_feature_fns(10), moment_fn=moment_fn,
             min_var_leaf_on_val=True, min_impurity_decrease=0.001,
             max_samples=.65, inference=False, subforest_size=1, random_state=123)
    ],
    param_grid_list=[
        {'alpha': np.geomspace(1e-6, 1e3, 20)},
        {'alpha': np.geomspace(1e-6, 1e3, 20)},
        {'alpha': np.geomspace(1e-6, 1e3, 20), 'l1_ratio': [.5, 1.0]},
        {'alpha': np.geomspace(1e-6, 1e3, 20), 'l1_ratio': [.5, 1.0]},
        {'riesz_feature_fns': [poly_feature_fns(2), poly_feature_fns(3)],
         'min_samples_leaf': [10, 50], 'min_var_fraction_leaf': [0.01, 0.1],
         'l2': [1e-5, 1e-3], 'max_depth': [5, None]}
    ],
    scoring=neg_reisz_score_gen(moment_fn), cv=cv, verbose=0, n_jobs=-1)


def rr_fn_gen(moment_fn, numeric_columns, cv=None, verbose=0):
    def get_rr_fn(X):
        est = first_stage_rr(moment_fn, numeric_columns, cv=cv).fit(X)
        if verbose > 0:
            print('Chose: ', est.best_score_, est.best_estimator_)
        return lambda: clone(est.best_estimator_)
    return get_rr_fn


class RegRandomForestClassifier(BaseEstimator):
    def __init__(self, *, max_depth=None, min_samples_leaf=1):
        self.max_depth = max_depth
        self.min_samples_leaf = min_samples_leaf

    def fit(self, X, y):
        self.model_ = RandomForestClassifier(n_estimators=100, random_state=123,
                                              max_depth=self.max_depth,
                                              min_samples_leaf=self.min_samples_leaf)
        self.model_.fit(X, y)
        return self

    def predict(self, X):
        return self.model_.predict_proba(X)[:, 1]


class RegLogisticRegression(BaseEstimator):
    def __init__(self, *, C=1):
        self.C = C

    def fit(self, X, y):
        self.model_ = LogisticRegression(C=self.C, max_iter=1000, random_state=123)
        self.model_.fit(X, y)
        return self

    def predict(self, X):
        return self.model_.predict_proba(X)[:, 1]

In [ ]:
def moment_estimate(est, inds=None):
    if inds is None:
        inds = np.arange(est.moment_reisz_.shape[0])
    theta = np.mean(est.moment_[inds])
    psi_theta = est.moment_[inds] - theta
    return theta, np.sqrt(np.mean(psi_theta**2) / psi_theta.shape[0])


def sensitivity_bounds_single(est, eta_ysq, eta_asq, alpha=None, inds=None):
    if inds is None:
        inds = np.arange(est.moment_reisz_.shape[0])
    nusq = np.mean(2 * est.moment_reisz_[inds] - est.reisz_pred_[inds]**2)
    sigmasq = np.mean((est.y_[inds] - est.reg_pred_[inds])**2)
    S = np.sqrt(sigmasq * nusq)
    Casq = eta_asq / (1 - eta_asq)
    Cgsq = eta_ysq
    theta = np.mean(est.moment_[inds])
    error = S * np.sqrt(Casq * Cgsq)
    if alpha is None:
        return theta - error, theta + error
    psi_theta = est.moment_[inds] - theta
    psi_sigmasq = (est.y_[inds] - est.reg_pred_[inds])**2 - sigmasq
    psi_nusq = 2 * est.moment_reisz_[inds] - est.reisz_pred_[inds]**2 - nusq
    phi_plus = psi_theta + (np.sqrt(Casq * Cgsq) / (2 * S)) * (sigmasq * psi_nusq + nusq * psi_sigmasq)
    stderr_plus = np.sqrt(np.mean(phi_plus**2) / phi_plus.shape[0])
    phi_minus = psi_theta - (np.sqrt(Casq * Cgsq) / (2 * S)) * (sigmasq * psi_nusq + nusq * psi_sigmasq)
    stderr_minus = np.sqrt(np.mean(phi_minus**2) / phi_minus.shape[0])
    return theta - error, stderr_minus, theta + error, stderr_plus


def sensitivity_bounds(est_list, eta_ysq, eta_asq, alpha=None, inds=None):
    if alpha is None:
        lower, upper = zip(*[sensitivity_bounds_single(est, eta_ysq, eta_asq,
                                                        alpha=alpha, inds=inds) for est in est_list])
        return np.mean(lower), np.mean(upper)
    else:
        lower, std_lower, upper, std_upper = zip(
            *[sensitivity_bounds_single(est, eta_ysq, eta_asq, alpha=alpha, inds=inds)
              for est in est_list])
        std_lower = np.array(std_lower)
        std_upper = np.array(std_upper)
        lower = np.mean(lower) - scipy.stats.norm.ppf(1 - alpha) * np.sqrt(np.mean(std_lower**2) + np.var(lower))
        upper = np.mean(upper) + scipy.stats.norm.ppf(1 - alpha) * np.sqrt(np.mean(std_upper**2) + np.var(upper))
        return lower, upper


def tvalue(est_list, value=0, leq=None, alpha=None, inds=None):
    for t in np.linspace(0, 1, 1000):
        l, u = sensitivity_bounds(est_list, t, t, alpha=alpha, inds=inds)
        if (leq is None) & (value >= l) & (value <= u):
            break
        if (leq == True) & (value >= l):
            break
        if (leq == False) & (value <= u):
            break
    return t


def sensitivity_contours(est_list, a_upper, y_upper, alpha=None, inds=None):
    xlist = np.linspace(0, a_upper, 100)
    ylist = np.linspace(0, y_upper, 100)
    X, Y = np.meshgrid(xlist, ylist)
    Zlower = np.zeros(X.shape)
    Zupper = np.zeros(X.shape)
    for itx in np.arange(X.shape[1]):
        for ity in np.arange(X.shape[0]):
            l, u = sensitivity_bounds(est_list, Y[ity, itx], X[ity, itx], alpha=alpha, inds=inds)
            Zlower[ity, itx] = l
            Zupper[ity, itx] = u
    return X, Y, Zlower, Zupper


def dml_estimate(est, inds=None):
    if inds is None:
        inds = np.arange(est.residuals_[0].shape[0])
    yres, Tres, _, _ = est.residuals_
    yres, Tres = yres[inds], Tres[inds]
    nusq = np.mean(Tres ** 2)
    theta = np.mean(yres * Tres) / nusq
    psi_theta = (yres - Tres * theta) * Tres / nusq
    return theta, np.sqrt(np.mean(psi_theta**2) / psi_theta.shape[0])


def dml_sensitivity_bounds_single(est, eta_ysq, eta_asq, alpha=None, inds=None):
    if inds is None:
        inds = np.arange(est.residuals_[0].shape[0])
    yres, Tres, _, _ = est.residuals_
    yres, Tres = yres[inds], Tres[inds]
    nusq = np.mean(Tres ** 2)
    theta = np.mean(yres * Tres) / nusq
    sigmasq = np.mean((yres - Tres * theta)**2)
    S = np.sqrt(sigmasq / nusq)
    Casq = eta_asq / (1 - eta_asq)
    Cgsq = eta_ysq
    error = S * np.sqrt(Casq * Cgsq)
    if alpha is None:
        return theta - error, theta + error
    psi_theta = (yres - Tres * theta) * Tres / nusq
    psi_sigmasq = (yres - Tres * theta)**2 - sigmasq
    psi_nusq = Tres**2 - nusq
    phi_plus = psi_theta + (np.sqrt(Casq * Cgsq) / (2 * S)) * (
        -(sigmasq / (nusq**2)) * psi_nusq + (1 / nusq) * psi_sigmasq)
    stderr_plus = np.sqrt(np.mean(phi_plus**2) / phi_plus.shape[0])
    phi_minus = psi_theta - (np.sqrt(Casq * Cgsq) / (2 * S)) * (
        -(sigmasq / (nusq**2)) * psi_nusq + (1 / nusq) * psi_sigmasq)
    stderr_minus = np.sqrt(np.mean(phi_minus**2) / phi_minus.shape[0])
    return theta - error, stderr_minus, theta + error, stderr_plus


def dml_sensitivity_bounds(est_list, eta_ysq, eta_asq, alpha=None, inds=None):
    if alpha is None:
        lower, upper = zip(*[dml_sensitivity_bounds_single(est, eta_ysq, eta_asq,
                                                            alpha=alpha, inds=inds)
                             for est in est_list])
        return np.mean(lower), np.mean(upper)
    else:
        lower, std_lower, upper, std_upper = zip(
            *[dml_sensitivity_bounds_single(est, eta_ysq, eta_asq, alpha=alpha, inds=inds)
              for est in est_list])
        std_lower = np.array(std_lower)
        std_upper = np.array(std_upper)
        lower = np.mean(lower) - scipy.stats.norm.ppf(1 - alpha) * np.sqrt(np.mean(std_lower**2) + np.var(lower))
        upper = np.mean(upper) + scipy.stats.norm.ppf(1 - alpha) * np.sqrt(np.mean(std_upper**2) + np.var(upper))
        return lower, upper


def dml_tvalue(est, value=0, leq=None, alpha=None, inds=None):
    for t in np.linspace(0, 1, 1000):
        l, u = dml_sensitivity_bounds(est, t, t, alpha=alpha, inds=inds)
        if (leq is None) & (value >= l) & (value <= u):
            break
        if (leq == True) & (value >= l):
            break
        if (leq == False) & (value <= u):
            break
    return t


def dml_sensitivity_contours(est_list, a_upper, y_upper, alpha=None, inds=None):
    xlist = np.linspace(0, a_upper, 100)
    ylist = np.linspace(0, y_upper, 100)
    X, Y = np.meshgrid(xlist, ylist)
    Zlower = np.zeros(X.shape)
    Zupper = np.zeros(X.shape)
    for itx in np.arange(X.shape[1]):
        for ity in np.arange(X.shape[0]):
            l, u = dml_sensitivity_bounds(est_list, Y[ity, itx], X[ity, itx], alpha=alpha, inds=inds)
            Zlower[ity, itx] = l
            Zupper[ity, itx] = u
    return X, Y, Zlower, Zupper

## Setting

The data come from the 2001 National Household Travel Survey and contain household gasoline consumption, gasoline prices, income brackets, and 24 observed controls describing population density, urbanization, demographics, and Census regions. After the sample restrictions used in the earlier demand studies, the analysis contains 3,466 households.

The outcome $Y$ is log gasoline consumption, the treatment $D$ is log gasoline price, and $X^o$ collects the observed controls. Under conditional ignorability, the short regression is

$$g_s(d, x^o) = \mathrm{E}[Y \mid D = d, X^o = x^o],$$

and the target is the average causal derivative

$$\theta_s = \mathrm{E}\big\{\partial_d\, g_s(D, X^o)\big\}.$$

Because both variables are in logarithms, $\theta_s$ is the average price elasticity of gasoline demand. Its short Riesz representer is the negative conditional density score,

$$\alpha_s(D, X^o) = -\partial_d \log f_{D \mid X^o}(D \mid X^o).$$

The moment function below encodes $m(W_s, g) = \partial_d\, g(D, X^o)$. The derivative with respect to log price is approximated by a central finite difference, $\{g(D + \epsilon, X^o) - g(D - \epsilon, X^o)\}/(2\epsilon)$ with $\epsilon = 0.01$.

In [ ]:
# The moment function, whose average we want to estimate: E[m(X; g)]
def moment_fn(x, test_fn):
    epsilon = 0.01
    t1 = np.hstack([x[:, [0]] + epsilon, x[:, 1:]])
    t0 = np.hstack([x[:, [0]] - epsilon, x[:, 1:]])
    return (test_fn(t1) - test_fn(t0)) / (2*epsilon)

## Data

Following the earlier demand studies, households with very low prices or incomes are dropped. Besides the full sample (`q=5`), the analysis uses four income groups. The first three (`q=1,2,3`) are overlapping groups centered near annual incomes of \$42,500, \$57,500 and \$72,500. The fourth (`q=4`) contains high-income households with income above roughly \$110,000 ($\log y \geq 11.6$).

Cross-validation and cross-fitting are grouped by state, so households from the same state remain in the same fold. The folds are stratified by Census region, and region dummies are included as controls.

In [ ]:
# Data loading and manipulation

# for semi-synthetic data generation
def true_f(X):
    y = (-.7 - .4 * scipy.special.expit(3 * (1 - 2*(X[:, 1] - 10.5)))) * X[:, 0]
    y += scipy.special.expit(3 * (1 - 2 * (X[:, 1] > 11.5)))
    return y


def get_data(q=None, synthetic=False, region=True, state=False, region_ids=None, random_seed=123):
    data_url = "https://raw.githubusercontent.com/CausalAIBook/MetricsMLNotebooks/update-notebooks/data/data_BHP2_with_region.csv"
    df = pd.read_csv(data_url)
    # Throw away low prices and low income due to sample quality as in Chetverikov
    df = df[df['log_p'] > math.log(1.2)]
    df = df[df['log_y'] > math.log(15000)]
    if not ((q == 4) | (q == 5)):  # all data in first three brackets
        df = df[df['log_y'] >= math.log(42500) - .5]
        df = df[df['log_y'] <= math.log(72500) + .5]
    if q == 1:
        df = df[df['log_y'] >= math.log(42500) - .5]
        df = df[df['log_y'] <= math.log(42500) + .5]
    if q == 2:
        df = df[df['log_y'] >= math.log(57500) - .5]
        df = df[df['log_y'] <= math.log(57500) + .5]
    if q == 3:
        df = df[df['log_y'] >= math.log(72500) - .5]
        df = df[df['log_y'] <= math.log(72500) + .5]
    if q == 4:
        df = df[df['log_y'] >= 11.6]
    if region_ids is not None:
        df = df[np.any([df['region'] == r for r in region_ids], axis=0)]
    Xdf = df.iloc[:, 1:]  # Covariates
    X_nostatedum = Xdf.drop(['distance_oil1000', 'share'], axis=1).values
    if state:
        state_dum = pd.get_dummies(Xdf['state_fips'], prefix='state', drop_first=True)
        Xdf = pd.concat([Xdf, state_dum], axis=1)
    if region:
        region_dum = pd.get_dummies(Xdf['region'], prefix='region', drop_first=True)
        Xdf = pd.concat([Xdf, region_dum], axis=1)
    Xdf = Xdf.drop(['distance_oil1000', 'state_fips', 'share', 'region'], axis=1)
    column_names = Xdf.columns
    X = Xdf.values
    y = df['log_q'].values
    if synthetic:
        y = true_f(X) + np.random.normal(0, .2, size=X[:, 0].shape)
        print(np.mean(moment_fn(X, true_f)))
    cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=random_seed)
    splits = list(cv.split(df, y=df['region'], groups=df['state_fips']))
    return X, y, [0, 1, 6, 7, 8], splits, column_names

In [ ]:
# global result dictionaries and global parameters
ests = {}
res = {}
res_d = {}
q_grid = np.arange(1, 6)
synthetic = False  # use synthetic outcome
verbose = 0
region = True  # control for region

## Short Estimates Under Conditional Ignorability

In [ ]:
# General pipeline for all types of estimators
def generic_pipeline(estimator, q_grid, name, n_seeds=5):
    ests = {}
    res = {}
    for q in q_grid:
        print(f'q={q}')
        ests[f'q={q}'] = []
        estimate_list = []
        for random_seed in np.arange(123, 123 + n_seeds):
            X, y, num_cols, splits, column_names = get_data(q, region=region, synthetic=synthetic,
                                                            random_seed=random_seed)
            estimates, model = estimator(X, y, num_cols, splits)
            print(estimates)
            estimate_list.append(estimates)
            ests[f'q={q}'].append(model)
        estimate_list = np.array(estimate_list)
        res[f'q={q}'] = {}
        res[f'q={q}'][f'{name}'] = {}
        res[f'q={q}'][f'{name}']['point'] = np.mean(estimate_list[:, 0])
        if estimate_list.shape[1] > 2:
            res[f'q={q}'][f'{name}']['stderr'] = np.sqrt(np.mean(estimate_list[:, 1]**2)
                                                          + np.var(estimate_list[:, 0]))
        res[f'q={q}'] = pd.DataFrame(res[f'q={q}']).transpose()
    return pd.concat(res), ests

In [ ]:
# Format results as a printable dataframe
def transform_res_to_df(res, q_grid, alpha=None):
    res_f = res.reset_index()
    res_d = {}
    for q in q_grid:
        key = 'all <= 11.6' if q == 0 else ('all' if q == 5 else f'q={q}')
        res_d[key] = {}
        for method in np.unique(res_f['level_1'].values):
            p = res.loc[(f'q={q}', method), 'point']
            if alpha is not None:
                s = res.loc[(f'q={q}', method), 'stderr']
                error = scipy.stats.norm.ppf(1 - alpha) * s
                l, u = p - error, p + error
                res_d[key][method] = f"{p:.2f} ({s:.2f}) [{l:.2f}, {u:.2f}]"
            else:
                res_d[key][method] = f"{p:.2f}"
    return pd.DataFrame(res_d)

In [ ]:
if synthetic:
    def oracle_estimator(X, y, num_cols, splits):
        return (np.mean(moment_fn(X, true_f)),), None
    res['true'], _ = generic_pipeline(oracle_estimator, q_grid, 'true', n_seeds=1)
    res_d['true'] = transform_res_to_df(res['true'], q_grid)
    display(res_d['true'])

### Nonparametric Model

The first specification leaves the short regression $g_s(D, X^o)$ unrestricted and targets $\theta_s = \mathrm{E}\{\partial_d g_s(D, X^o)\}$ directly. Rather than estimating the conditional density derivative in $\alpha_s$, which can be unstable, we learn the Riesz representer from the Riesz loss. Model selection is used for both the regression and the representer. Candidate procedures include expanded-feature lasso, forests and local polynomial forests for the regression, and penalized linear Riesz regression or ForestRiesz for the representer. The target and score stay fixed while the approximation class changes.

In [ ]:
# fully non-parametric average derivative estimation
def nonlinrr(X, y, num_cols, splits):
    get_reisz_fn = rr_fn_gen(moment_fn, num_cols, cv=splits, verbose=verbose)
    get_reg_fn = reg_fn_gen(num_cols, cv=splits, verbose=verbose)
    est = DebiasedMoment(moment_fn=moment_fn,
                         get_reisz_fn=get_reisz_fn,
                         get_reg_fn=get_reg_fn,
                         n_splits=splits)
    est.fit(X, y)
    return est.avg_moment(alpha=.1), est

In [ ]:
res['rr'], ests['rr'] = generic_pipeline(nonlinrr, q_grid, 'dr')

In [ ]:
res_d['rr'] = transform_res_to_df(res['rr'], q_grid, alpha=0.05)
res_d['rr']

### Partially Linear Model

The second specification is the partially linear model,

$$Y = \theta_s D + f_s(X^o) + \varepsilon, \qquad \mathrm{E}[\varepsilon \mid D, X^o] = 0,$$

in which the elasticity is constant.

In [ ]:
# partially linear average derivative estimation, with doubleML
def dml_estimator(X, y, num_cols, splits):
    num_cols = [t - 1 for t in num_cols[1:]]
    model_y = reg_fn_gen(num_cols, cv=splits, verbose=verbose)(X[:, 1:], y)
    model_t = reg_fn_gen(num_cols, cv=splits, verbose=verbose)(X[:, 1:], X[:, 0])
    est = LinearDML(model_y=model_y(),
                    model_t=model_t(),
                    linear_first_stages=False,
                    cv=splits, random_state=123)
    est.fit(y, X[:, 0], W=X[:, 1:], cache_values=True)
    p = est.intercept__inference().point_estimate
    s = est.intercept__inference().stderr
    l, u = est.intercept__inference().conf_int(alpha=.1)
    return (p, s, l, u), est

In [ ]:
res['dml'], ests['dml'] = generic_pipeline(dml_estimator, q_grid, 'dml')

In [ ]:
res_d['dml'] = transform_res_to_df(res['dml'], q_grid, alpha=.05)
res_d['dml']

## Sensitivity Results for the Full Sample

For the full sample we report the short elasticity, the sensitivity bounds and their confidence bounds under the scenario $C_Y^2 = 0.03$, $C_D^2 \approx 0.03$ and $|\rho| = 1$, together with robustness values.

In [ ]:
table = {}
table['point'] = '{:.3f} ({:.3f})'.format(res['rr'].loc[(f'q={5}', 'dr'), 'point'],
                                           res['rr'].loc[(f'q={5}', 'dr'), 'stderr'])

est_list = ests['rr'][f'q={5}']
table['Bounds'] = '[{:.3f}, {:.3f}]'.format(*sensitivity_bounds(est_list, .03, .03, alpha=None))
table['CIBounds'] = '[{:.3f}, {:.3f}]'.format(*sensitivity_bounds(est_list, .03, .03, alpha=.05))
table['RV(v=-1.5)'] = '{:.3f}'.format(tvalue(est_list, value=-1.5, leq=True, alpha=None))
table['RV(v=-1.5,a=.05)'] = '{:.3f}'.format(tvalue(est_list, value=-1.5, leq=True, alpha=.05))
table['RV(v=0)'] = '{:.3f}'.format(tvalue(est_list, value=0, leq=False, alpha=None))
table['RV(v=0,a=.05)'] = '{:.3f}'.format(tvalue(est_list, value=0, leq=False, alpha=.05))

In [ ]:
table2 = {}
table2['point'] = '{:.3f} ({:.3f})'.format(res['dml'].loc[(f'q={5}', 'dml'), 'point'],
                                            res['dml'].loc[(f'q={5}', 'dml'), 'stderr'])

est_list = ests['dml'][f'q={5}']
table2['Bounds'] = '[{:.3f}, {:.3f}]'.format(*dml_sensitivity_bounds(est_list, .03, .03, alpha=None))
table2['CIBounds'] = '[{:.3f}, {:.3f}]'.format(*dml_sensitivity_bounds(est_list, .03, .03, alpha=.05))
table2['RV(v=-1.5)'] = '{:.3f}'.format(dml_tvalue(est_list, value=-1.5, leq=True, alpha=None))
table2['RV(v=-1.5,a=.05)'] = '{:.3f}'.format(dml_tvalue(est_list, value=-1.5, leq=True, alpha=.05))
table2['RV(v=0)'] = '{:.3f}'.format(dml_tvalue(est_list, value=0, leq=False, alpha=None))
table2['RV(v=0,a=.05)'] = '{:.3f}'.format(dml_tvalue(est_list, value=0, leq=False, alpha=.05))

In [ ]:
tabledf = pd.DataFrame({'Non-parametric': table, 'Partially linear': table2}).T[
    ['point', 'Bounds', 'CIBounds', 'RV(v=-1.5)', 'RV(v=-1.5,a=.05)', 'RV(v=0)', 'RV(v=0,a=.05)']]
tabledf

In [ ]:
qgr = np.arange(1, 5)

point = [res['rr'].loc[(f'q={q}', 'dr'), 'point'] for q in qgr]
lower = [res['rr'].loc[(f'q={q}', 'dr'), 'point'] - 1.645*res['rr'].loc[(f'q={q}', 'dr'), 'stderr']
         for q in qgr]
upper = [res['rr'].loc[(f'q={q}', 'dr'), 'point'] + 1.645*res['rr'].loc[(f'q={q}', 'dr'), 'stderr']
         for q in qgr]
sens_band = [sensitivity_bounds(ests['rr'][f'q={q}'], .03, .03, alpha=None) for q in qgr]
sens_band_unc = [sensitivity_bounds(ests['rr'][f'q={q}'], .03, .03, alpha=.05) for q in qgr]

if synthetic:
    true_moment = [res['true'].loc[(f'q={q}', 'true'), 'point'] for q in qgr]

plt.figure(figsize=(5, 5))
plt.plot(qgr, point)
plt.fill_between(qgr, lower, upper, alpha=.4, label='5%-95% CI')
plt.fill_between(qgr, *list(zip(*sens_band)), alpha=.6, label='Bounds')
plt.fill_between(qgr, *list(zip(*sens_band_unc)), alpha=.2, label='5%-95% Bounds')
if synthetic:
    plt.plot(qgr, true_moment, label='true')
plt.xticks(qgr)
plt.title('Non-parametric')
plt.legend()
plt.grid(color='0.85', linewidth=0.6)

In [ ]:
qgr = np.arange(1, 5)

point = [res['dml'].loc[(f'q={q}', 'dml'), 'point'] for q in qgr]
lower = [res['dml'].loc[(f'q={q}', 'dml'), 'point'] - 1.645*res['dml'].loc[(f'q={q}', 'dml'), 'stderr']
         for q in qgr]
upper = [res['dml'].loc[(f'q={q}', 'dml'), 'point'] + 1.645*res['dml'].loc[(f'q={q}', 'dml'), 'stderr']
         for q in qgr]
sens_band = [dml_sensitivity_bounds(ests['dml'][f'q={q}'], .03, .03, alpha=None) for q in qgr]
sens_band_unc = [dml_sensitivity_bounds(ests['dml'][f'q={q}'], .03, .03, alpha=.05) for q in qgr]

plt.figure(figsize=(5, 5))
plt.plot(qgr, point)
plt.fill_between(qgr, lower, upper, alpha=.4, label='5%-95% CI')
plt.fill_between(qgr, *list(zip(*sens_band)), alpha=.6, label='Bounds')
plt.fill_between(qgr, *list(zip(*sens_band_unc)), alpha=.2, label='5%-95% Bounds')
if synthetic:
    plt.plot(qgr, true_moment, label='true')
plt.xticks(qgr)
plt.title('Partially linear')
plt.legend()
plt.grid(color='0.85', linewidth=0.6)